# Compare `MyNordic_auxiliary` and Dynawo Nordic `LoadFlow`

This notebook runs both Modelica cases and compares:
- the number of buses, non-slack generators, and transformers
- the voltage angle `UPhase` and reactive power `Q` for each non-slack generator
- the side-1 voltage angle and side-1 reactive power for each transformer
- the active and reactive power of the slack
- the top 5 biggest discrepancies across non-slack generators and transformers

Notes:
- For transformers there is no single built-in `UPhase` variable, so this notebook uses the angle of the side-1 terminal voltage.


In [1]:
using OMJulia
using DataFrames

# --- Configuration ---

# Local root for the OpenModelica-only notebooks
DEFAULT_ROOT_DIR = "/home/clarafercas/dynawo-notebooks/OpenModelica_only_users"
ROOT_DIR = isdir(joinpath(DEFAULT_ROOT_DIR, "BuildAux")) ? DEFAULT_ROOT_DIR : pwd()

# Generated auxiliary package to validate
AUX_PACKAGE_DIR = joinpath(ROOT_DIR, "BuildAux", "MyNordic_auxiliary")
AUX_PACKAGE_FILE = joinpath(AUX_PACKAGE_DIR, "package.mo")
AUX_MODEL = "MyNordic_auxiliary.TestCase_auxiliary"

# Dynawo reference model
REFERENCE_PACKAGE_FILE = "/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo"
REFERENCE_MODEL = "Dynawo.Examples.Nordic.TestCases.LoadFlow"

# Libraries
MODELICA_PKG_PATH = "/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo"
DYNAWO_PKG_PATH = REFERENCE_PACKAGE_FILE

# Comparison settings
STOP_TIME = 1.0
MY_SLACK = "g20"
REFERENCE_SLACK = "slackbus_g20"


"slackbus_g20"

In [2]:
function om_send(omc, expr; parsed = true)
    println("OMC -> ", expr)
    try
        return sendExpression(omc, expr; parsed = parsed)
    catch err
        println(sendExpression(omc, "getErrorString()", parsed = false))
        rethrow(err)
    end
end

function get_result_variable_names(omc, resultfile::String)
    vars = sendExpression(omc, "readSimulationResultVars(\"$resultfile\")")
    return sort!(String.(vars))
end

function collect_component_names_from_results(omc, resultfile::String, suffix::String; prefix = nothing, pattern = nothing)
    names = String[]
    seen = Set{String}()

    for var in get_result_variable_names(omc, resultfile)
        endswith(var, suffix) || continue
        name = chopsuffix(var, suffix)
        !isnothing(prefix) && !startswith(name, prefix) && continue
        !isnothing(pattern) && !occursin(pattern, name) && continue
        name in seen && continue
        push!(seen, name)
        push!(names, name)
    end

    sort!(names)
    return names
end

function read_last_value(sys, full_name::String)
    values = getSolutions(sys, full_name)
    series = values[1]
    isempty(series) && error("No values found for $full_name")
    return Float64(series[end])
end

function read_complex(sys, base_name::String)
    re = read_last_value(sys, base_name * ".re")
    im = read_last_value(sys, base_name * ".im")
    return complex(re, im)
end

function voltage_angle_deg_from_connector(sys, connector_path::String)
    v = read_complex(sys, connector_path * ".V")
    return rad2deg(atan(imag(v), real(v)))
end

function terminal_power_pu(sys, component::String)
    v = read_complex(sys, "$component.terminal.V")
    i = read_complex(sys, "$component.terminal.i")
    s = v * conj(i)
    return real(s), imag(s)
end

function run_and_simulate(label::String, package_file::String, model_name::String; stop_time::Float64 = STOP_TIME)
    isfile(package_file) || error("Package file not found: $package_file")

    omc = OMJulia.OMCSession()
    om_send(omc, "loadFile(\"$MODELICA_PKG_PATH\")")
    om_send(omc, "loadModel(Complex)")
    om_send(omc, "loadModel(ModelicaServices)")
    om_send(omc, "loadFile(\"$DYNAWO_PKG_PATH\")")
    package_file == DYNAWO_PKG_PATH || om_send(omc, "loadFile(\"$package_file\")")

    println("\nChecking $label...")
    chk = om_send(omc, "checkModel($model_name)", parsed = false)
    println(chk)

    build_dir = mktempdir()
    resultfile_name = replace(lowercase(label), " " => "_") * "_res.mat"

    ModelicaSystem(
        omc,
        package_file,
        model_name,
        [MODELICA_PKG_PATH, DYNAWO_PKG_PATH],
        customBuildDirectory = build_dir,
    )

    setSimulationOptions(omc, "stopTime=$(stop_time)")
    simulate(omc, resultfile = resultfile_name)
    resultfile_path = joinpath(getWorkDirectory(omc), resultfile_name)

    return Dict(
        "label" => label,
        "omc" => omc,
        "resultfile" => resultfile_path,
        "build_dir" => build_dir,
        "model" => model_name,
    )
end

function build_count_table(my_bus_count::Int, ref_bus_count::Int, my_generator_count::Int, ref_generator_count::Int, my_transformer_count::Int, ref_transformer_count::Int)
    return DataFrame(
        component_type = ["buses", "non-slack generators", "transformers"],
        mynordic_count = [my_bus_count, my_generator_count, my_transformer_count],
        dynawo_count = [ref_bus_count, ref_generator_count, ref_transformer_count],
        difference = [my_bus_count - ref_bus_count, my_generator_count - ref_generator_count, my_transformer_count - ref_transformer_count],
    )
end

function empty_generator_uphase_q_table()
    return DataFrame(
        generator_name = String[],
        mynordic_uphase_deg = Float64[],
        dynawo_uphase_deg = Float64[],
        delta_uphase_deg = Float64[],
        mynordic_q_pu = Float64[],
        dynawo_q_pu = Float64[],
        delta_q_pu = Float64[],
    )
end

function build_generator_uphase_q_table(my_sys, ref_sys, generators::Vector{String})
    rows = NamedTuple[]
    for gen in sort(generators)
        my_uphase = voltage_angle_deg_from_connector(my_sys, "$gen.terminal")
        ref_uphase = voltage_angle_deg_from_connector(ref_sys, "$gen.terminal")
        my_q = read_last_value(my_sys, "$gen.QGenPu")
        ref_q = read_last_value(ref_sys, "$gen.QGenPu")

        push!(rows, (
            generator_name = gen,
            mynordic_uphase_deg = my_uphase,
            dynawo_uphase_deg = ref_uphase,
            delta_uphase_deg = my_uphase - ref_uphase,
            mynordic_q_pu = my_q,
            dynawo_q_pu = ref_q,
            delta_q_pu = my_q - ref_q,
        ))
    end
    return isempty(rows) ? empty_generator_uphase_q_table() : DataFrame(rows)
end

function empty_transformer_uphase_q_table()
    return DataFrame(
        transformer_name = String[],
        mynordic_uphase_side1_deg = Float64[],
        dynawo_uphase_side1_deg = Float64[],
        delta_uphase_deg = Float64[],
        mynordic_q1_pu = Float64[],
        dynawo_q1_pu = Float64[],
        delta_q_pu = Float64[],
    )
end

function build_transformer_uphase_q_table(my_sys, ref_sys, transformers::Vector{String})
    rows = NamedTuple[]
    for trafo in sort(transformers)
        my_uphase = voltage_angle_deg_from_connector(my_sys, "$trafo.terminal1")
        ref_uphase = voltage_angle_deg_from_connector(ref_sys, "$trafo.terminal1")
        my_q = read_last_value(my_sys, "$trafo.Q1Pu")
        ref_q = read_last_value(ref_sys, "$trafo.Q1Pu")

        push!(rows, (
            transformer_name = trafo,
            mynordic_uphase_side1_deg = my_uphase,
            dynawo_uphase_side1_deg = ref_uphase,
            delta_uphase_deg = my_uphase - ref_uphase,
            mynordic_q1_pu = my_q,
            dynawo_q1_pu = ref_q,
            delta_q_pu = my_q - ref_q,
        ))
    end
    return isempty(rows) ? empty_transformer_uphase_q_table() : DataFrame(rows)
end

function empty_top_component_discrepancy_table()
    return DataFrame(
        component_type = String[],
        component_name = String[],
        mynordic_uphase_deg = Float64[],
        dynawo_uphase_deg = Float64[],
        delta_uphase_deg = Float64[],
        abs_delta_uphase_deg = Float64[],
        mynordic_q_pu = Float64[],
        dynawo_q_pu = Float64[],
        delta_q_pu = Float64[],
        abs_delta_q_pu = Float64[],
    )
end

function build_top_component_discrepancy_table(generator_df::DataFrame, transformer_df::DataFrame; n::Int = 5)
    rows = NamedTuple[]

    for row in eachrow(generator_df)
        push!(rows, (
            component_type = "generator",
            component_name = row.generator_name,
            mynordic_uphase_deg = row.mynordic_uphase_deg,
            dynawo_uphase_deg = row.dynawo_uphase_deg,
            delta_uphase_deg = row.delta_uphase_deg,
            abs_delta_uphase_deg = abs(row.delta_uphase_deg),
            mynordic_q_pu = row.mynordic_q_pu,
            dynawo_q_pu = row.dynawo_q_pu,
            delta_q_pu = row.delta_q_pu,
            abs_delta_q_pu = abs(row.delta_q_pu),
        ))
    end

    for row in eachrow(transformer_df)
        push!(rows, (
            component_type = "transformer",
            component_name = row.transformer_name,
            mynordic_uphase_deg = row.mynordic_uphase_side1_deg,
            dynawo_uphase_deg = row.dynawo_uphase_side1_deg,
            delta_uphase_deg = row.delta_uphase_deg,
            abs_delta_uphase_deg = abs(row.delta_uphase_deg),
            mynordic_q_pu = row.mynordic_q1_pu,
            dynawo_q_pu = row.dynawo_q1_pu,
            delta_q_pu = row.delta_q_pu,
            abs_delta_q_pu = abs(row.delta_q_pu),
        ))
    end

    isempty(rows) && return empty_top_component_discrepancy_table()

    df = DataFrame(rows)
    sort!(df, [:abs_delta_q_pu, :abs_delta_uphase_deg], rev = [true, true])
    return first(df, min(n, nrow(df)))
end

function build_slack_pq_table(my_sys, ref_sys, my_slack::String, ref_slack::String)
    my_p, my_q = terminal_power_pu(my_sys, my_slack)
    ref_p, ref_q = terminal_power_pu(ref_sys, ref_slack)

    return DataFrame(
        mynordic_slack = [my_slack],
        dynawo_slack = [ref_slack],
        mynordic_p_pu = [my_p],
        dynawo_p_pu = [ref_p],
        delta_p_pu = [my_p - ref_p],
        mynordic_q_pu = [my_q],
        dynawo_q_pu = [ref_q],
        delta_q_pu = [my_q - ref_q],
    )
end


build_slack_pq_table (generic function with 1 method)

In [3]:
my_run = run_and_simulate("MyNordic auxiliary", AUX_PACKAGE_FILE, AUX_MODEL; stop_time = STOP_TIME)
ref_run = run_and_simulate("Dynawo Nordic LoadFlow", REFERENCE_PACKAGE_FILE, REFERENCE_MODEL; stop_time = STOP_TIME)

my_omc = my_run["omc"]
ref_omc = ref_run["omc"]

println("\nMyNordic result file: ", my_run["resultfile"])
println("Reference result file: ", ref_run["resultfile"])


[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.IYy4IKVNuj"


OMC -> loadFile("/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo")
OMC -> loadModel(Complex)
OMC -> loadModel(ModelicaServices)
OMC -> loadFile("/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo")
OMC -> loadFile("/home/clarafercas/dynawo-notebooks/OpenModelica_only_users/BuildAux/MyNordic_auxiliary/package.mo")

Checking MyNordic auxiliary...
OMC -> checkModel(MyNordic_auxiliary.TestCase_auxiliary)
"Check of MyNordic_auxiliary.TestCase_auxiliary completed successfully.
Class MyNordic_auxiliary.TestCase_auxiliary has 5819 equation(s) and 5819 variable(s).
1550 of these are trivial equation(s)."

stopTime=1.0

OMC -> loadFile("/home/clarafercas/dynawo/OpenModelica/lib/omlibrary/Modelica/package.mo")


[ Info: Path to zmq file="/tmp/openmodelica.clarafercas.port.julia.JLG6tSzCwT"


OMC -> loadModel(Complex)
OMC -> loadModel(ModelicaServices)
OMC -> loadFile("/home/clarafercas/Model_library/dynawo/dynawo/sources/Models/Modelica/Dynawo/package.mo")

Checking Dynawo Nordic LoadFlow...
OMC -> checkModel(Dynawo.Examples.Nordic.TestCases.LoadFlow)
"Check of Dynawo.Examples.Nordic.TestCases.LoadFlow completed successfully.
Class Dynawo.Examples.Nordic.TestCases.LoadFlow has 3181 equation(s) and 3181 variable(s).
1143 of these are trivial equation(s)."

stopTime=1.0


MyNordic result file: /tmp/jl_uzCv4u/mynordic_auxiliary_res.mat
Reference result file: /tmp/jl_1Q0aWF/dynawo_nordic_loadflow_res.mat


In [4]:
my_buses = collect_component_names_from_results(my_omc, my_run["resultfile"], ".terminal.V.re"; prefix = "bus_")
ref_buses = collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".terminal.V.re"; prefix = "bus_")

my_generators = setdiff(collect_component_names_from_results(my_omc, my_run["resultfile"], ".QGenPu"; pattern = r"^g\d+$"), [MY_SLACK])
ref_generators = setdiff(collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".QGenPu"; pattern = r"^g\d+$"), [REFERENCE_SLACK])
common_generators = sort(intersect(my_generators, ref_generators))

my_transformers = collect_component_names_from_results(my_omc, my_run["resultfile"], ".Q1Pu"; prefix = "trafo_")
ref_transformers = collect_component_names_from_results(ref_omc, ref_run["resultfile"], ".Q1Pu"; prefix = "trafo_")
common_transformers = sort(intersect(my_transformers, ref_transformers))

count_comparison_df = build_count_table(
    length(my_buses),
    length(ref_buses),
    length(my_generators),
    length(ref_generators),
    length(my_transformers),
    length(ref_transformers),
)

generator_uphase_q_df = build_generator_uphase_q_table(my_omc, ref_omc, common_generators)
transformer_uphase_q_df = build_transformer_uphase_q_table(my_omc, ref_omc, common_transformers)
top_component_discrepancy_df = build_top_component_discrepancy_table(generator_uphase_q_df, transformer_uphase_q_df)
slack_pq_df = build_slack_pq_table(my_omc, ref_omc, MY_SLACK, REFERENCE_SLACK)


Row,mynordic_slack,dynawo_slack,mynordic_p_pu,dynawo_p_pu,delta_p_pu,mynordic_q_pu,dynawo_q_pu,delta_q_pu
,String,String,Float64,Float64,Float64,Float64,Float64,Float64
1,g20,slackbus_g20,-21.374,-21.374,1.42109e-14,-3.77386,-3.77386,3.10862e-15


In [5]:
println("Counts in both models:")
display(count_comparison_df)

println("Top 5 discrepancies across non-slack generators and transformers (sorted by |delta Q|, then |delta UPhase|):")
display(top_component_discrepancy_df)

println("Non-slack generators: UPhase and Q in both models:")
display(generator_uphase_q_df)

println("Transformers: side-1 voltage angle and side-1 Q in both models:")
display(transformer_uphase_q_df)

println("Slack P and Q in both models:")
display(slack_pq_df)


Counts in both models:


Row,component_type,mynordic_count,dynawo_count,difference
,String,Int64,Int64,Int64
1,buses,74,74,0
2,non-slack generators,19,19,0
3,transformers,50,50,0


Top 5 discrepancies across non-slack generators and transformers (sorted by |delta Q|, then |delta UPhase|):


Row,component_type,component_name,mynordic_uphase_deg,dynawo_uphase_deg,delta_uphase_deg,abs_delta_uphase_deg,mynordic_q_pu,dynawo_q_pu,delta_q_pu,abs_delta_q_pu
,String,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,generator,g14,-49.9006,-49.9006,7.10543e-14,7.10543e-14,2.95855,2.95855,8.50024e-8,8.50024e-8
2,transformer,trafo_g14_4042,-49.9006,-49.9006,7.10543e-14,7.10543e-14,2.95855,2.95855,8.50024e-8,8.50024e-8
3,generator,g17,-46.8548,-46.8548,9.23706e-14,9.23706e-14,0.487272,0.487272,7.83409e-8,7.83409e-8
4,transformer,trafo_g17_4062,-46.8548,-46.8548,9.23706e-14,9.23706e-14,0.487272,0.487272,7.83409e-8,7.83409e-8
5,transformer,trafo_1044_4044a,-67.7101,-67.7101,8.52651e-14,8.52651e-14,-0.0736793,-0.0736792,-6.48319e-8,6.48319e-8


Non-slack generators: UPhase and Q in both models:


Row,generator_name,mynordic_uphase_deg,dynawo_uphase_deg,delta_uphase_deg,mynordic_q_pu,dynawo_q_pu,delta_q_pu
,String,Float64,Float64,Float64,Float64,Float64,Float64
1,g01,2.58511,2.58511,2.9754e-14,0.583425,0.583425,-8.43769e-15
2,g02,5.11636,5.11636,3.01981e-14,0.172375,0.172375,-2.84512e-9
3,g03,10.2744,10.2744,2.84217e-14,0.209155,0.209155,1.1366e-8
4,g04,8.02718,8.02718,3.90799e-14,0.303899,0.303899,-4.52844e-9
5,g05,-12.3588,-12.3588,4.26326e-14,0.600891,0.600891,2.05953e-8
6,g06,-59.4198,-59.4198,9.9476e-14,1.38571,1.38571,3.70309e-8
7,g07,-68.9537,-68.9537,8.52651e-14,0.604206,0.604206,1.0151e-8
8,g08,-16.8146,-16.8146,5.32907e-14,2.32593,2.32593,-3.46591e-8
9,g09,-1.62843,-1.62843,3.13083e-14,2.01277,2.01277,3.56035e-8


Transformers: side-1 voltage angle and side-1 Q in both models:


Row,transformer_name,mynordic_uphase_side1_deg,dynawo_uphase_side1_deg,delta_uphase_deg,mynordic_q1_pu,dynawo_q1_pu,delta_q_pu
,String,Float64,Float64,Float64,Float64,Float64,Float64
1,trafo_1011_4011,-6.64741,-6.64741,3.19744e-14,-1.90487,-1.90487,-4.83308e-8
2,trafo_1012_4012,-3.09768,-3.09768,3.10862e-14,-1.7357,-1.7357,-4.97565e-9
3,trafo_1022_4022,-19.0452,-19.0452,4.9738e-14,-1.55727,-1.55727,-3.33067e-15
4,trafo_1044_4044a,-67.7101,-67.7101,8.52651e-14,-0.0736793,-0.0736792,-6.48319e-8
5,trafo_1044_4044b,-67.7101,-67.7101,8.52651e-14,-0.0736793,-0.0736792,-6.26361e-8
6,trafo_1045_4045a,-71.6628,-71.6628,8.52651e-14,-0.0570757,-0.0570757,-4.44089e-16
7,trafo_1045_4045b,-71.6628,-71.6628,8.52651e-14,-0.0570757,-0.0570757,-2.22435e-8
8,trafo_11_1011,-9.44737,-9.44737,3.19744e-14,-0.688,-0.688,0.0
9,trafo_12_1012,-5.93463,-5.93463,3.01981e-14,-0.838,-0.838,-2.14148e-8


Slack P and Q in both models:


Row,mynordic_slack,dynawo_slack,mynordic_p_pu,dynawo_p_pu,delta_p_pu,mynordic_q_pu,dynawo_q_pu,delta_q_pu
,String,String,Float64,Float64,Float64,Float64,Float64,Float64
1,g20,slackbus_g20,-21.374,-21.374,1.42109e-14,-3.77386,-3.77386,3.10862e-15
